# Hermes Agent: Procedural Memory (스킬 학습) 실습

`03_hermes_memory_architecture.ipynb`에서 Working → Episodic → Semantic 메모리를 다뤘습니다. 이 노트북은 마지막 계층인 **Procedural Memory**를 다룹니다. 에이전트가 일하면서 **"이 사용자의 이런 작업은 이렇게 한다"는 절차를 스스로 스킬로 기록**하고, 다음 세션에서 그 스킬을 꺼내 쓰는 학습 루프입니다.

구현은 오픈소스 Hermes Agent의 소스(`agent/background_review.py`, `tools/skill_manager_tool.py`, `tools/skills_tool.py`, `tools/skill_usage.py`, `agent/curator.py`)를 읽고, 핵심 학습 루프는 같게 두고 운영용 기능은 덜어낸 교육용 재구성입니다. 덜어낸 기능은 Part 8에서 설명합니다.

---

### 🎯 학습 목표
1. **Semantic과 Procedural의 차이**: "무엇을/누구를" 기억하는 것과 "어떻게"를 기억하는 것이 왜 다른 저장소가 되는지 이해합니다.
2. **스킬 저장소와 Progressive Disclosure**: 프롬프트에는 이름과 설명만 두고, 본문은 `skill_view`로 필요할 때 여는 구조를 확인합니다.
3. **`memory_snapshot` 주입 내용**: 한 턴이 시작될 때 세션 동결 스냅샷(`AgentContext.memory_snapshot`)에 실제로 어떤 텍스트가 들어가는지 직접 출력해 봅니다.
4. **학습 트리거**: 턴 수가 아니라 **도구 호출 반복 횟수**로 스킬 리뷰를 시작하는 이유를 확인합니다.
5. **백그라운드 스킬 리뷰**: 스킬 도구만 가진 리뷰 에이전트가 대화를 다시 보고 스킬을 만들거나 고치는 과정을 실행합니다.
6. **전이 실험**: 세션 A에서 배운 절차가 세션 B의 다른 데이터에서 재사용되는지 측정합니다.
7. **자기 개선과 생애주기**: 교정이 기존 스킬의 patch로 반영되는 과정과, 쓰이지 않는 스킬이 보관되는 과정을 확인합니다.

### 🗂️ 구성
| Part | 내용 | LLM 호출 |
|:---|:---|:---|
| 0 | 4계층 메모리 지도와 Hermes 소스 대응표 | 없음 |
| 1 | `ProceduralMemoryStore` — 스킬 생성, 보조 파일, 검증 | 없음 |
| 2 | Progressive Disclosure, Frozen Snapshot, **`memory_snapshot`에 실제로 들어가는 내용** | 없음 |
| 3 | 학습 트리거 — 도구 호출 반복 카운터 | 없음 (가짜 모델) |
| 4 | 백그라운드 스킬 리뷰와 자율 쓰기 가드 | 있음 |
| 5 | 전이 실험: 세션 A에서 학습 → 세션 B에서 재사용 | 있음 |
| 6 | 자기 개선: 교정이 기존 스킬의 patch로 반영 | 있음 |
| 7 | Curator — 사용 기록 기반 생애주기 | 없음 |
| 8 | 교육용으로 생략한 Hermes 운영 기능 | 없음 |
| 9 | FAQ, Clean-up | 없음 |

### 🧩 이 노트북에서 자주 나오는 이름
| 이름 | 종류 | 한 줄 설명 |
|:---|:---|:---|
| `ProceduralMemoryStore` | 클래스 | 스킬 폴더를 읽고 쓰는 저장소. 검증과 가드를 담당하고 **LLM은 쓰지 않음** |
| `skill_view` / `skill_manage` | 에이전트 도구 | 모델이 스킬을 열람·생성·수정할 때 호출하는 도구. 내부에서 저장소 메서드를 부름 |
| `run_skill_review` | 함수 | **LLM 리뷰 에이전트**를 실행해 "무엇을 배웠고 어느 스킬을 어떻게 고칠지" 판단 |
| `ReviewSession` | 클래스 | "이 쓰기는 사용자가 없는 백그라운드 리뷰가 한다"는 **권한 표시**. 이번 리뷰에서 읽은 파일 목록을 들고 다님 |
| `MemoryMiddleware` | 미들웨어 | 매 턴 메모리를 인출(`memory_snapshot` / `episodic_context`)하고, 도구 호출 수를 세고, 기준값에 닿은 턴이 끝나면 리뷰를 실행 |
| `memory_snapshot` | 필드 | `AgentContext`의 필드. 세션 첫 턴에 동결된 장기기억 스냅샷(`MEMORY.md` + `USER.md` + `PROCEDURAL` 카탈로그 → Message 3) |

## 🛠️ Step 0. 환경 세팅

이 노트북은 실제 앱과 같은 사용자별 경로를 씁니다. 실습 사용자는 `temporary_user`이고, 학습 결과는 아래에 저장됩니다.

```
artifacts/users/temporary_user/
├── memory/   MEMORY.md, USER.md       ← Semantic (에이전트가 memory 도구로 쓰면 여기에 기록)
└── skills/   <name>/SKILL.md, .usage.json  ← Procedural (이 노트북의 주인공)
```

- 이 셀을 실행할 때마다 `temporary_user` 폴더를 비우고 새로 시작합니다.
- 실습용 CSV 데이터만 `artifacts/temporary/procedural_lab/`에 만들고 마지막 셀에서 지웁니다. 학습 결과(`temporary_user`)는 남겨 두므로, 노트북을 끝낸 뒤 파일을 직접 열어 볼 수 있습니다.
- 두 폴더 모두 `.gitignore`에 들어 있어 커밋되지 않습니다.
- LLM은 프로젝트 기본 모델(`configs/model.config`)을 씁니다. 모델에 따라 결과가 달라지는 이유는 Part 5-5에서 설명합니다.

In [ ]:
import os
import sys
import json
import shutil
import difflib
import warnings
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
nest_asyncio.apply()   # 주피터의 이벤트 루프 안에서 asyncio 코드를 다시 실행할 수 있게 함

# ── 1. 프로젝트 루트를 찾아 작업 디렉터리와 import 경로로 설정 ──
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
load_dotenv(os.path.join(project_root, ".env"), override=True)   # API 키 로드
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# ── 2. 이 노트북에서 쓰는 모듈 ──
from app.utils import init_chat_model, normalize_content
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.checkpoint.memory import MemorySaver

from app.context import AgentContext
from app.middleware.memory import (
    ProceduralMemoryStore,   # 스킬 저장소 (LLM 없음)
    ReviewSession,           # 백그라운드 쓰기 권한 표시
    SemanticMemoryStore,     # MEMORY.md / USER.md 저장소
    MemoryMiddleware,        # 메모리 주입 + 트리거 + 백그라운드 리뷰
    run_skill_review,        # LLM 스킬 리뷰 에이전트 실행
    SKILL_REVIEW_PROMPT,     # 리뷰 에이전트의 시스템 프롬프트
)
from app.utils.workspace import USERS_ROOT_DIR, resolve_user_memory_dir, resolve_user_skills_dir

# ── 3. 실습 사용자 폴더 준비: artifacts/users/temporary_user/ ──
USER_ID = "temporary_user"
USER_DIR = USERS_ROOT_DIR / USER_ID
shutil.rmtree(USER_DIR, ignore_errors=True)                       # 다시 실행할 때 이전 학습 결과 초기화
SKILLS_DIR = resolve_user_skills_dir(USER_ID, ensure_exists=True)   # artifacts/users/temporary_user/skills

# ── 4. 실습용 데이터 파일(CSV 등)만 두는 임시 폴더 ──
SANDBOX = Path("./artifacts/temporary/procedural_lab").resolve()
shutil.rmtree(SANDBOX, ignore_errors=True)
SANDBOX.mkdir(parents=True)

# ── 5. LLM: 프로젝트 기본 모델(configs/model.config) ──
with open("./configs/model.config", encoding="utf-8") as f:
    MODEL_NAME = json.load(f).get("model_name", "gemini-3.8-flash")
llm = init_chat_model(model=MODEL_NAME, temperature=0.0)

print(f"✅ Project Root: {project_root}")
print(f"✅ 실습 사용자: {USER_ID} → {USER_DIR.relative_to(project_root)}")
print(f"✅ 실습 데이터: {SANDBOX.relative_to(project_root)}")
print(f"✅ LLM: {MODEL_NAME}")

## Part 0. 4계층 메모리 지도

| 계층 | 기억하는 것 | 저장소 | 프롬프트 주입 | 학습 시점 |
|:---|:---|:---|:---|:---|
| L1 Working | 지금 대화 | 체크포인터 (`checkpoints.db`) | 대화 그 자체 | 매 턴 |
| L2 Episodic | 지난 세션에 있었던 일 | `transcript.db` 원문 + `episodic.db` 요약 | 관련 세션 요약 힌트 (`episodic_context` → 대화 끝 `<system-reminder>`) | 세션 종료 시 에피소드 추출 |
| L3 Semantic | **무엇을 / 누구를** — 사용자 정보, 환경 사실 | `MEMORY.md` / `USER.md` | 전문 (`memory_snapshot` → Message 3, Frozen Snapshot) | 백그라운드 리뷰 (사용자 턴 nudge) |
| **Procedural** | **어떻게** — 작업 유형별 절차, 함정, 결과물 선호 | `skills/<name>/SKILL.md` | **이름 + 설명만** (`memory_snapshot` → Message 3, Frozen Snapshot) | 백그라운드 리뷰 (도구 호출 nudge) |

> L3와 Procedural 내용은 세션 첫 턴에 동결되어 **`AgentContext.memory_snapshot`**(Message 3 `# memorySnapshot`)에 함께 들어가고, L2 에피소드 검색 결과는 매 턴 **`AgentContext.episodic_context`**(대화 끝 `<system-reminder>`)로 들어갑니다. 실제 모습은 Part 2-4에서 출력해 봅니다.

### Semantic과 Procedural을 왜 나누는가
Hermes의 메모리 안내문(`agent/prompt_builder.py`의 `build_memory_guidance`)은 둘의 경계를 이렇게 정합니다.

> Skills come first: when you learn something while doing a task — a procedure, a pitfall, and the user's preferences and corrections for that kind of work — record it in the skill. **Memory is the narrow exception for facts that apply to EVERY session regardless of task.**

- "사용자는 데이터 분석가이고 한국어로 보고서를 쓴다"처럼 **모든 세션에 적용되는 사실**은 Semantic(`USER.md`)에 둡니다.
- "정합성 보고서는 손상 파일이면 백업을 쓰고, 첫 줄에 결론, 표는 마크다운으로"처럼 **특정 작업 유형에만 필요한 절차**는 스킬에 둡니다. 스킬은 해당 작업을 할 때만 열리므로, 항상 프롬프트에 실리는 Semantic의 글자 수 예산을 차지하지 않습니다.

### Hermes 소스 대응표
| 구성 요소 | Hermes | 이 저장소 |
|:---|:---|:---|
| 스킬 저장·검증 | `tools/skill_manager_tool.py` | `app/middleware/memory/procedural_store.py` |
| 스킬 열람·목록 | `tools/skills_tool.py` | `procedural_store.py` (`view`, `list_skills`) |
| 사용 기록·출처 | `tools/skill_usage.py` (`.usage.json`) | `procedural_store.py` (`.usage.json`) |
| 자율 쓰기 가드 | `tools/skill_manager_guards.py` | `procedural_store.py` (`ReviewSession`) |
| 도구 스키마 | `skill_view`, `skill_manage`, `skills_list` | `app/middleware/memory/skill_tools.py` |
| 학습 트리거 | `_iters_since_skill` (`turn_iteration_prep.py`, `turn_finalizer.py`) | `MemoryMiddleware._count_skill_iteration` |
| 스킬 리뷰 | `agent/background_review.py` (`_SKILL_REVIEW_PROMPT`, 리뷰 포크) | `app/middleware/memory/skill_review.py` |
| 생애주기 | `agent/curator.py` | `procedural_store.py` (`apply_lifecycle`) |

## Part 1. `ProceduralMemoryStore` — 스킬 하나의 구조

스킬은 폴더 하나입니다. `SKILL.md`에는 **항상 지켜야 할 규칙**을, `references/`에는 **가끔만 필요한 깊이 있는 자료**를 둡니다.

```
skills/
├── .usage.json              ← 사용 기록 + 출처 (SKILL.md 본문과 분리)
└── weekly-status-email/
    ├── SKILL.md             ← frontmatter(name, description) + 절차 본문
    ├── references/          ← 주제별 참고 자료
    ├── templates/           ← 복사해서 고쳐 쓰는 시작 파일
    └── scripts/             ← 그대로 다시 실행하는 스크립트
```

`description`의 첫 문장은 **언제 이 스킬을 열어야 하는지(trigger)**를 말해야 합니다. 프롬프트에는 이 설명만 실리기 때문입니다.

### 1-1. 스킬 만들기와 보조 파일 추가
**왜 필요한가**: 저장소가 스킬을 어떤 모양으로 저장하는지 먼저 손으로 확인합니다. 나중에 LLM 리뷰가 하는 일도 결국 이 메서드들(`create`, `write_file`, `patch`)을 부르는 것입니다.

**볼 것**: 메서드마다 돌려주는 결과 딕셔너리, 그리고 `.usage.json`에 기록된 `created_by` 값.

In [ ]:
# 실습 사용자의 스킬 라이브러리를 엽니다 (폴더가 비어 있으면 그대로 빈 라이브러리).
store = ProceduralMemoryStore(skills_dir=str(SKILLS_DIR))   # artifacts/users/temporary_user/skills
store.load_from_disk()

# SKILL.md 전체 텍스트: YAML frontmatter(name, description) + 마크다운 본문
skill_text = """---
name: weekly-status-email
description: Use when writing the weekly team status email. Lead with blockers, then progress.
---
# Weekly Status Email

## Procedure
1. Collect this week's merged work and open blockers.
2. Write blockers first, each with an owner and a date.
3. Summarize progress in at most five bullets. Tone guide: references/tone.md

## Pitfalls
- Do not list routine meetings — readers scan the email for decisions, not attendance.
"""

# (1) 스킬 생성 — review 인자를 주지 않았으므로 "사용자가 시킨 포그라운드 쓰기"로 기록됩니다.
print("=== 1) create (포그라운드 = 사용자 지시) ===")
result = store.create("weekly-status-email", skill_text)
print(json.dumps(result, ensure_ascii=False, indent=2))

# (2) 보조 파일 추가 — 경로는 반드시 references/ templates/ scripts/ assets/ 중 하나로 시작
print("\n=== 2) write_file — 보조 파일 추가 ===")
result = store.write_file(
    "weekly-status-email", "references/tone.md", "- Plain sentences, no emoji.\n- Numbers over adjectives."
)
print(json.dumps(result, ensure_ascii=False, indent=2))

# (3) 사용 기록 파일(.usage.json) — 스킬 본문과 분리된 메타데이터
print("\n=== 3) 출처 기록 (.usage.json) ===")
print(json.dumps(store.load_usage(), ensure_ascii=False, indent=2))

`created_by`가 `null`입니다. **사용자가 시켜서 포그라운드에서 만든 스킬은 사용자 소유**라서, 나중에 백그라운드 리뷰가 자율적으로 고칠 수 없습니다. 백그라운드 리뷰가 스스로 만든 스킬만 `created_by: "agent"`가 됩니다(Part 4).

`patch_count`가 1인 것은 (2)의 보조 파일 추가가 스킬 수정으로 집계되었기 때문입니다.

### 1-2. 검증 — 잘못된 쓰기는 저장소가 거부합니다
**왜 필요한가**: 스킬 파일은 LLM이 씁니다. 모델이 이름을 잘못 짓거나, frontmatter를 빠뜨리거나, `../`로 다른 폴더에 파일을 쓰려 해도 라이브러리가 망가지지 않아야 합니다. 그래서 저장소가 모양을 강제합니다.

**볼 것**: 7가지 잘못된 요청이 모두 ❌로 거부되고, 오류 메시지가 **모델이 다음에 무엇을 해야 하는지** 알려 주는지.

In [ ]:
# 일부러 잘못된 요청 7가지를 보냅니다. 람다로 감싸 두고 하나씩 실행합니다.
cases = {
    "이름 규칙 위반 (대문자/공백)": lambda: store.create("Weekly Email", skill_text),
    "frontmatter 누락": lambda: store.create("no-frontmatter", "# Title\nbody"),
    "frontmatter 이름 불일치": lambda: store.create("another-name", skill_text),       # 본문 name은 weekly-status-email
    "같은 이름 중복 생성": lambda: store.create("weekly-status-email", skill_text),
    "경로 탈출 (../)": lambda: store.write_file("weekly-status-email", "../../evil.md", "x"),
    "허용되지 않은 하위 폴더": lambda: store.write_file("weekly-status-email", "notes/x.md", "x"),
    "patch 대상 문자열이 여러 곳": lambda: store.patch("weekly-status-email", "blockers", "risks"),  # 'blockers'가 3번 나옴
}
for label, fn in cases.items():
    r = fn()
    mark = "✅" if r["success"] else "❌"
    print(f"{mark} {label:<26} → {r.get('error', r.get('message'))[:110]}")

## Part 2. Progressive Disclosure와 Frozen Snapshot

### 2-1. 프롬프트에는 카탈로그만
Hermes는 스킬 본문을 프롬프트에 넣지 않습니다. 이름과 설명만 담긴 목록(`<available_skills>`)을 주고, 관련 있는 스킬은 모델이 `skill_view`로 엽니다. 스킬이 수십 개가 되어도 프롬프트가 커지지 않습니다.

### 2-2. 카탈로그는 세션 시작 시점에 고정
Semantic Memory와 같은 **Frozen Snapshot** 규칙입니다. 세션 도중 스킬이 새로 생겨도 프롬프트는 바뀌지 않고, **다음 세션에서** 반영됩니다. 세션 중 프롬프트 앞부분이 바뀌지 않아야 KV 캐시가 유지되기 때문입니다.

**왜 아래 셀이 필요한가**: Part 1에서 스킬을 만들었지만, 저장소를 연 시점(`load_from_disk`)에는 스킬이 0개였습니다. 다시 로드하기 전과 후의 카탈로그를 비교하면 "세션 중 불변, 다음 세션에 반영"이 눈에 보입니다.

**볼 것**: 첫 출력은 `None`(스냅샷 당시 스킬 없음), 두 번째 출력에는 `weekly-status-email` 한 줄.

In [ ]:
# format_for_prompt(): 프롬프트에 실릴 카탈로그 텍스트 (스냅샷)
print("=== 지금 프롬프트에 실릴 카탈로그 (load 시점엔 스킬 0개) ===")
print(store.format_for_prompt())

# 새 세션이 시작되면 MemoryMiddleware가 load_from_disk()를 다시 불러 스냅샷을 새로 찍습니다.
store.load_from_disk()
print("\n=== 새 세션에서 다시 로드한 카탈로그 ===")
print(store.format_for_prompt())

### 2-3. `skill_view`로 본문 열기
**왜 필요한가**: 카탈로그에서 스킬을 고른 모델은 `skill_view(name)` 도구를 호출합니다. 그때 모델이 받는 결과가 무엇인지 확인합니다. 도구는 내부에서 `store.view()`를 부르므로, 여기서는 저장소 메서드를 직접 호출합니다.

**볼 것**
- 반환값에 본문(`content`)과 함께 `linked_files`가 있어서, 모델이 보조 파일의 존재를 알 수 있습니다.
- `file_path`를 주면 보조 파일만 엽니다.
- 열람할 때마다 `view_count`, `use_count`, `last_used_at`이 갱신됩니다. Curator(Part 7)는 이 시각으로 "오래 안 쓴 스킬"을 판단합니다.

In [ ]:
# (1) SKILL.md 열기 — 모델이 skill_view(name="weekly-status-email")을 호출한 것과 같음
view = store.view("weekly-status-email")
print("=== skill_view(name) 반환값 (content 제외) ===")
print(json.dumps({k: v for k, v in view.items() if k != "content"}, ensure_ascii=False, indent=2))
print("\n=== view['content'] — SKILL.md 본문 ===")
print(view["content"])

# (2) 보조 파일 열기 — 모델이 skill_view(name=..., file_path="references/tone.md")를 호출한 것과 같음
ref = store.view("weekly-status-email", file_path="references/tone.md")
print("=== skill_view(name, file_path) — 보조 파일 ===")
print(ref["content"])

# (3) 열람 기록 확인 — 위에서 두 번 열었으므로 2
usage = store.load_usage()["weekly-status-email"]
print("\n=== 열람은 '사용'으로 집계 (Curator가 이 시각으로 stale 판단) ===")
print("view_count  :", usage["view_count"])
print("use_count   :", usage["use_count"])
print("last_used_at:", usage["last_used_at"])

### 2-4. `memory_snapshot`에 실제로 들어가는 내용
한 턴이 시작될 때 `MemoryMiddleware.before_agent` 훅은 프리픽스 캐시 보존을 위해 세션 동결 스냅샷(`AgentContext.memory_snapshot`)과 턴별 에피소드 기억(`AgentContext.episodic_context`)을 분리해 기록합니다(하위 호환용 합본 `recalled_memory`도 함께 채움). 실제 앱(`PromptAssembler` Claude Code 모드)에서는 L4(`[9] memory`)에는 메모리 사용 규칙(`# auto memory`)만 고정하고, `memory_snapshot`(MEMORY/USER/PROCEDURAL 카탈로그)은 **Message 3(`# memorySnapshot`)**에, `episodic_context`는 **대화 끝 `<system-reminder>`(`# episodicMemory`)**에 분리 주입합니다.

| 컨텍스트 변수 | 블록 | 출처 | 내용 |
|:---|:---|:---|:---|
| `memory_snapshot` | `MEMORY (agent's personal notes)` | L3 `MEMORY.md` | 에이전트가 기록한 환경·프로젝트 사실 **전문** |
| `memory_snapshot` | `USER PROFILE (who the user is)` | L3 `USER.md` | 사용자에 대한 사실 **전문** |
| `memory_snapshot` | `PROCEDURAL MEMORY (learned skills)` | `skills/` | 안내문 + **스킬 이름과 설명 목록** (본문 없음) |
| `episodic_context` | `EPISODIC MEMORY (relevant past sessions)` | L2 `episodic.db` | 이번 질문과 관련된 과거 세션 요약 (검색 결과가 있을 때만) |

> 사람이 작성한 번들 스킬(`app/agents/main_agent/skills/`의 금융 스킬 등)은 `memory_snapshot`이 아니라 **대화 속 스킬 목록 알림(`skill_listing`, 노트북 02 셀 4)**으로 따로 전달됩니다. `memory_snapshot`의 목록은 **이 사용자에게 학습된 스킬**만입니다.

먼저 이후 Part에서도 계속 쓸 **조립 함수 두 개**를 정의합니다.
- `make_memory_mw()`: 실제 앱(`main_agent/agent.py`)과 같은 방식으로 `MemoryMiddleware`를 만듭니다. 기본 스토어는 `default` 사용자 경로에 두고, 요청마다 `AgentContext.user_id`를 보고 `artifacts/users/{user_id}/` 아래 스토어를 고릅니다.
- `lab_context()`: 실습용 요청 컨텍스트입니다. 사용자를 `temporary_user`로 지정하고, 스킬 학습만 관찰하도록 Semantic 학습과 Episodic은 끕니다.

In [ ]:
def make_memory_mw(interval, review_llm):
    """실제 앱과 같은 구성의 MemoryMiddleware를 만듭니다.

    interval  : 스킬 리뷰를 시작할 도구 호출 반복 횟수 (Hermes 기본 10)
    review_llm: 백그라운드 리뷰에 쓸 LLM (None이면 리뷰 없음)
    """
    # 기본 스토어 = default 사용자 경로. 실제 사용자 폴더는 요청의 user_id로 자동 선택됩니다.
    # 글자 수 상한도 main_agent와 같게 둡니다 (MEMORY.md 4,000자 / USER.md 2,000자).
    sem = SemanticMemoryStore(memory_dir=str(resolve_user_memory_dir("default")),
                              memory_char_limit=4000, user_char_limit=2000)
    sem.load_from_disk()
    proc = ProceduralMemoryStore(skills_dir=str(resolve_user_skills_dir("default")))
    proc.load_from_disk()
    return MemoryMiddleware(
        semantic_store=sem,
        episodic_store=None,              # 이 노트북은 Procedural에 집중 (Episodic은 03 참고)
        review_llm=review_llm,
        transcript_enabled=False,         # 원문 저장소(transcript.db)는 쓰지 않음
        finalize_idle_settle_s=3600.0,    # 세션 종료(에피소드 추출) 대기. 스킬 리뷰는 턴 종료 시 바로 실행
        procedural_store=proc,
        skill_nudge_interval=interval,
    )


def lab_context(session_id):
    """실습용 AgentContext: user_id=temporary_user → artifacts/users/temporary_user/{memory,skills}"""
    return AgentContext(
        session_id=session_id,
        user_id=USER_ID,
        semantic_learning_enabled=False,  # USER.md/MEMORY.md 자동 학습은 끔 (스킬 학습만 관찰)
        episodic_memory_enabled=False,    # 과거 세션 요약 검색도 끔
        transcript_enabled=False,
    )

print("✅ make_memory_mw(), lab_context() 정의 완료")

**왜 아래 셀이 필요한가**: 실제 에이전트를 돌리지 않고도 `before_agent` 훅만 직접 호출하면, 모델이 받게 될 `memory_snapshot` 텍스트를 그대로 꺼내 볼 수 있습니다. Semantic 블록에도 내용이 보이도록 `temporary_user`의 `USER.md`에 사실 하나를 넣고 시작합니다.

**볼 것**
- USER PROFILE 블록에 방금 넣은 사실이 **전문**으로 들어갑니다.
- PROCEDURAL MEMORY 블록에는 Part 1의 스킬이 **이름과 설명 한 줄**로만 들어가고 본문은 없습니다.
- Episodic은 꺼 두었으므로 `ctx.episodic_context`는 비어 있습니다.

In [ ]:
from types import SimpleNamespace

l4_mw = make_memory_mw(interval=10, review_llm=None)

# (1) temporary_user의 USER.md에 사실 하나 추가 → artifacts/users/temporary_user/memory/USER.md
l4_mw._get_semantic_store(USER_ID).add("user", "Data analyst who writes reports in Korean.")

# (2) 새 턴이 시작될 때 LangGraph가 훅에 넘겨주는 runtime을 흉내 냅니다 (context + config).
ctx = lab_context("l4-demo")
runtime = SimpleNamespace(context=ctx, config={"configurable": {"thread_id": "l4-demo"}})

# (3) before_agent 훅 실행 → ctx.memory_snapshot에 세션 동결 스냅샷(L3 + 학습 스킬 카탈로그)이 채워집니다.
l4_mw.before_agent({"messages": [HumanMessage(content="이번 주 상태 메일 써줘")]}, runtime)

print("=== AgentContext.memory_snapshot (세션 동결 스냅샷 → Message 3 # memorySnapshot) ===")
print(ctx.memory_snapshot)

l4_mw._review_queue.shutdown()   # 이 데모용 미들웨어의 백그라운드 큐 정리

## Part 3. 학습 트리거 — 왜 "도구 호출 횟수"인가

Hermes는 메모리와 스킬의 리뷰 트리거가 다릅니다.

| | 트리거 | 이유 |
|:---|:---|:---|
| Memory (Semantic) | 사용자 **턴** 수 (`_turns_since_memory`, 기본 10) | 사용자에 대한 사실은 대화량에 비례해 드러남 |
| **Skill (Procedural)** | **도구를 호출한 모델 응답** 수 (`_iters_since_skill`, 기본 10) | 절차는 **여러 단계를 거친 작업**에서 생김. 잡담 100턴에는 배울 절차가 없음 |

이 저장소도 같은 구조입니다. Semantic 리뷰는 `memory_nudge_interval`(사용자 턴, Hermes 기본 10 / 이 저장소의 교육용 설정 1 — 노트북 03 Part 3), 스킬 리뷰는 `skill_nudge_interval`(도구 호출 응답, 기본 10)로 따로 셉니다. 아래는 스킬 카운터의 규칙입니다.

규칙은 세 가지입니다.
1. 도구를 호출한 모델 응답 1회를 1로 셉니다. 한 응답에서 도구를 병렬로 여러 개 불러도 1입니다.
2. 에이전트가 스스로 `skill_manage`를 호출하면 0으로 리셋합니다. 이미 배운 것을 기록했기 때문입니다.
3. 기준값에 도달한 턴이 끝나면 곧바로 백그라운드 스킬 리뷰를 실행합니다(Hermes와 같음). 사용자가 새 메시지를 보내면 진행 중인 리뷰는 취소됩니다.

한 번 기준값에 도달한 세션은 **스킬 리뷰가 실제로 끝날 때까지 기준값 도달 표시를 유지**합니다. 새 턴 때문에 리뷰가 취소되면 카운터도 그대로 두고, 그 턴이 끝날 때 다시 실행합니다.

### 3-1. 가짜 모델 준비
**왜 필요한가**: 카운터가 정확히 몇이 되는지 확인하려면 모델이 **언제 어떤 도구를 부를지 정해져** 있어야 합니다. 실제 LLM은 매번 다르게 행동하므로, 정해진 응답을 순서대로 돌려주는 가짜 모델을 만듭니다. API 비용도 들지 않습니다.

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

class ScriptedModel(GenericFakeChatModel):
    """정해진 응답을 순서대로 돌려주는 가짜 모델."""
    def bind_tools(self, tools, **kwargs):
        return self   # create_agent가 도구를 바인딩하려 할 때 그냥 자기 자신을 돌려줌

def scripted(*msgs):
    """문자열이나 AIMessage를 받아, 그 순서대로 응답하는 가짜 모델을 만듭니다."""
    return ScriptedModel(messages=iter([m if isinstance(m, AIMessage) else AIMessage(content=m) for m in msgs]))

def tool_call(name, args, call_id):
    """도구 하나를 호출하는 모델 응답(AIMessage)을 만듭니다."""
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id, "type": "tool_call"}])

@tool
def lookup(query: str) -> str:
    """Look something up."""
    return f"result for {query}"   # 실험용 더미 도구

print("✅ 가짜 모델과 더미 도구 준비 완료")

### 3-2. 카운터 실험
**시나리오**: 가짜 모델이 한 턴 안에서 아래 순서로 응답합니다. 기준값은 3으로 낮춰 둡니다.
1. `lookup` 1개 호출
2. `lookup` 2개를 **병렬**로 호출 (한 응답)
3. `lookup` 1개 호출
4. 최종 답변 "끝" (도구 없음)

**볼 것**: 도구 실행은 4번이지만 카운터는 **3**입니다(도구를 부른 응답 수). 기준값 3에 도달했으므로 nudge 조건(`_skill_nudge_reached`)이 `True`가 됩니다. 이 셀은 리뷰 LLM을 주지 않아(`review_llm=None`) 조건만 확인하고, 실제 리뷰는 Part 4·5에서 실행합니다.

In [ ]:
# 리뷰 LLM 없이 카운터와 nudge 조건만 확인합니다 (review_llm이 있으면 이 턴이 끝나자마자 스킬 리뷰가 시작됨).
counter_mw = make_memory_mw(interval=3, review_llm=None)

parallel = AIMessage(content="", tool_calls=[            # 한 응답에서 도구 2개 병렬 호출
    {"name": "lookup", "args": {"query": "a"}, "id": "p1", "type": "tool_call"},
    {"name": "lookup", "args": {"query": "b"}, "id": "p2", "type": "tool_call"},
])
demo_agent = create_agent(
    model=scripted(tool_call("lookup", {"query": "x"}, "c1"), parallel, tool_call("lookup", {"query": "y"}, "c3"), "끝"),
    tools=[lookup],
    middleware=[counter_mw],
    checkpointer=MemorySaver(),
    context_schema=AgentContext,
)
demo_agent.invoke({"messages": [HumanMessage(content="조사해줘")]},
                  config={"configurable": {"thread_id": "count-demo"}}, context=lab_context("count-demo"))

# 미들웨어 내부 상태를 직접 들여다봅니다.
print("세션별 도구 호출 반복 횟수 :", counter_mw._skill_iters)          # {'count-demo': 3}
print("nudge 조건 충족 (_skill_nudge_reached):", counter_mw._skill_nudge_reached("count-demo"))  # True
counter_mw._review_queue.shutdown()

## Part 4. 백그라운드 스킬 리뷰

리뷰는 **스킬 도구 3종(`skills_list`, `skill_view`, `skill_manage`)만 가진 별도 에이전트**가 합니다. 메인 대화와 프롬프트 캐시에는 영향을 주지 않습니다.

```
run_skill_review(llm, store, 대화)              ← 판단 (LLM)
 └─ 리뷰 에이전트 (create_agent, 스킬 도구 3개만)
     ├─ skills_list   → store.list_skills()
     ├─ skill_view    → store.view(..., review=세션)    ← "읽었음" 기록
     └─ skill_manage  → store.create / store.patch(..., review=세션)   ← 실제 파일 수정 (LLM 없음)
```

### 리뷰 프롬프트의 핵심 원칙 (Hermes `_SKILL_REVIEW_PROMPT` 요약)
1. **클래스 단위**: 스킬 이름은 작업 유형이어야 합니다(`data-integrity-report`). 오늘의 파일명이나 오류 문자열(`fix-sales-csv-today`)이면 안 됩니다.
2. **기록이 아닌 교훈**: "이번에 무슨 일이 있었다"가 아니라 "앞으로 이렇게 하라 + 한 줄 이유"를 씁니다. 날짜, 티켓 번호, 사용자 발언 인용을 넣지 않습니다.
3. **고치기가 만들기보다 먼저**: 기존 스킬 patch → 보조 파일 추가 → 새 스킬 생성 순으로 고릅니다. 비슷한 스킬이 여러 개 생기는 것을 막습니다.
4. **사용자 교정은 1급 신호**: "이렇게 하지 마", "형식을 바꿔"는 그 작업을 다루는 스킬에 **기본 동작**으로 반영해야 다음 세션이 처음부터 맞게 합니다.
5. **저장하지 말 것**: 환경 문제(패키지 미설치 등), "도구 X는 안 된다" 같은 부정 주장, 재시도로 풀린 일시 오류, 일회성 작업, 끝내 해결하지 못한 실패.

**왜 아래 셀이 필요한가**: 리뷰 에이전트가 받는 지시 전문입니다. 위 원칙이 실제로 어떤 문장으로 쓰였는지 확인합니다.

In [ ]:
print(SKILL_REVIEW_PROMPT)

### 4-1. 대화 기록으로 리뷰 실행
**왜 필요한가**: 실제 에이전트를 돌리기 전에, 미리 준비한 대화 기록 하나로 리뷰 에이전트만 실행해 봅니다. 회의록 정리 대화이고, 도구 호출과 **사용자의 형식 교정**이 들어 있습니다. 형식은 `transcript.db`에 저장되는 원문과 같습니다(역할, 도구 호출, 도구 결과).

`run_skill_review()`가 돌려주는 딕셔너리:
| 키 | 뜻 |
|:---|:---|
| `actions` | 성공한 스킬 쓰기 요약 (예: `"Skill 'x' created"`) |
| `trace` | 리뷰 에이전트가 호출한 도구와 성공/실패 순서 |
| `reply` | 리뷰 에이전트의 마지막 응답 |
| `cancelled`, `stopped_early` | 새 턴 때문에 취소됐는지, 반복 상한에 걸렸는지 |

**볼 것**: 새 스킬이 생기고 `created_by=agent`로 기록되는지, 사용자 교정이 절차와 출력 형식으로 들어갔는지.

In [ ]:
# Part 1과 같은 temporary_user 스킬 라이브러리 (weekly-status-email이 이미 들어 있음)
review_store = ProceduralMemoryStore(skills_dir=str(SKILLS_DIR))
review_store.load_from_disk()

# 리뷰할 대화: 첫 결과 → 사용자 교정 → 다시 작성 → 사용자 확인
meeting_transcript = [
    {"role": "user", "content": "notes/sprint_review.txt 회의록을 정리해서 summary.md로 저장해줘."},
    {"role": "assistant", "content": "", "tool_calls": [{"name": "read_file", "args": {"path": "notes/sprint_review.txt"}}]},
    {"role": "tool", "tool_name": "read_file", "content": "[00:01] 민수: 배포 금요일로 확정 ... [00:20] 지은: QA는 목요일까지 ..."},
    {"role": "assistant", "content": "", "tool_calls": [{"name": "write_file", "args": {"path": "summary.md", "content": "회의 요약: 여러 논의가 있었음 ..."}}]},
    {"role": "tool", "tool_name": "write_file", "content": "saved"},
    {"role": "assistant", "content": "회의 내용을 시간순으로 요약해 저장했습니다."},
    {"role": "user", "content": "시간순 요약은 필요 없어. 결정 사항을 맨 위에 bullet로, 그다음 담당자·할 일·기한 표를 만들어. 잡담은 빼고."},  # ← 교정
    {"role": "assistant", "content": "", "tool_calls": [{"name": "write_file", "args": {"path": "summary.md", "content": "## 결정 사항\n- 배포 금요일\n## 할 일\n| 담당자 | 할 일 | 기한 | ..."}}]},
    {"role": "tool", "tool_name": "write_file", "content": "saved"},
    {"role": "assistant", "content": "결정 사항을 맨 위에 두고 담당자/할 일/기한 표로 다시 저장했습니다."},
    {"role": "user", "content": "좋아, 이 형식이 맞아."},
]

before_review = {s["name"] for s in review_store.list_skills()}   # 리뷰 전 스킬 이름
result = run_skill_review(llm, review_store, meeting_transcript)  # ← LLM 리뷰 에이전트 실행

print("=== 리뷰 결과 요약 (result['actions']) ===")
print(result["actions"] or "(변경 없음)")
print("\n=== 리뷰 에이전트의 도구 호출 순서 (result['trace']) ===")
for step in result["trace"]:
    print(" ", step)
print("\n=== 리뷰 에이전트의 마지막 응답 (result['reply']) ===")
print(result["reply"])

# 리뷰가 새로 만든 스킬만 골라 본문 출력
for s in review_store.list_skills():
    if s["name"] in before_review:
        continue
    print(f"\n=== 생성된 스킬: {s['name']} (created_by={s['created_by']}) ===")
    print((review_store.skills_dir / s["name"] / "SKILL.md").read_text(encoding="utf-8"))

확인할 점:
- 이름이 `summary-md`나 `sprint-review` 같은 이번 파일 이름이 아니라 회의록 정리라는 **작업 유형**을 가리키는지
- 사용자의 교정(결정 사항 먼저, 담당자/할 일/기한 표, 잡담 제외)이 **절차와 출력 선호**로 들어갔는지
- `created_by`가 `agent`인지. 이 스킬은 이후 백그라운드 리뷰가 고칠 수 있습니다.

### 4-2. 배울 것이 없는 대화
**왜 필요한가**: 리뷰가 매번 무언가를 만들면 라이브러리가 쓸모없는 스킬로 채워집니다. 일회성 질문(환율 한 번 묻기)에는 리뷰가 아무것도 저장하지 않아야 합니다.

**볼 것**: `actions`가 빈 목록이고 `reply`가 "Nothing to save."이며, 스킬 목록이 그대로인지.

In [ ]:
small_talk = [
    {"role": "user", "content": "1 달러는 몇 원이야?"},
    {"role": "assistant", "content": "", "tool_calls": [{"name": "convert_currency", "args": {"amount": 1, "from": "USD", "to": "KRW"}}]},
    {"role": "tool", "tool_name": "convert_currency", "content": "1 USD = 1,385 KRW"},
    {"role": "assistant", "content": "1달러는 약 1,385원입니다."},
]
before = [s["name"] for s in review_store.list_skills()]
result = run_skill_review(llm, review_store, small_talk)
print("actions:", result["actions"])
print("reply  :", result["reply"])
print("스킬 목록 변화:", before, "→", [s["name"] for s in review_store.list_skills()])

### 4-3. 자율 쓰기 가드
리뷰 에이전트는 사용자가 없는 상태에서 파일을 고칩니다. 그래서 Hermes는 두 가지 가드를 둡니다.

| 가드 | 규칙 | 막는 사고 |
|:---|:---|:---|
| 소유권 | `created_by: agent`인 스킬만 수정 | 사용자가 직접 쓴 스킬을 에이전트가 몰래 바꾸는 일 |
| Read-before-write | **이번 리뷰에서** `skill_view`로 읽은 파일만 수정 | 대화에 남은 옛 본문을 기준으로 patch해 최신 내용을 덮어쓰는 일 |

**왜 아래 셀이 필요한가**: 가드는 LLM이 아니라 저장소 코드에 있습니다. 그래서 LLM 없이, **리뷰 에이전트가 내부에서 하는 호출(`store.patch(..., review=세션)`)을 손으로 재현**해 가드가 어떻게 막는지 확인합니다.

`review=` 인자는 "리뷰를 실행하라"는 뜻이 아닙니다. **"이 쓰기는 사용자가 없는 백그라운드 리뷰가 하는 것"이라는 권한 표시**입니다. `ReviewSession`은 이번 리뷰에서 읽은 파일 목록만 기억합니다.
- `review=None` → 포그라운드(사용자 지시) 쓰기: 가드 없음
- `review=ReviewSession()` → 백그라운드 쓰기: 소유권 가드 + read-before-write 가드

**볼 것**: (1) 사용자 소유 스킬은 읽었어도 거부, (2) 에이전트 소유 스킬은 읽기 전에는 거부되고 `skill_view` 후에는 성공.

In [ ]:
# ── (1) 사용자 소유 스킬(Part 1에서 포그라운드로 생성) → 읽었어도 거부 ──
guard_review = ReviewSession()                                   # 백그라운드 리뷰 1회의 권한 표시
store.view("weekly-status-email", review=guard_review)           # 읽기는 허용 (읽었음 기록)
r = store.patch("weekly-status-email", "at most five bullets", "at most three bullets", review=guard_review)
print("=== (1) 사용자 소유 스킬 patch ===")
print(r["error"])

# ── (2) 에이전트 소유 스킬(4-1에서 리뷰가 생성) → 이번 리뷰에서 읽기 전에는 거부 ──
target = next((s["name"] for s in review_store.list_skills() if s["created_by"] == "agent"), None)
if target:
    fresh_review = ReviewSession()                               # 아직 아무것도 읽지 않은 새 리뷰
    body = (review_store.skills_dir / target / "SKILL.md").read_text(encoding="utf-8")
    # frontmatter 뒤 본문에서 고칠 줄 하나를 고릅니다 (번호 목록, bullet, 제목 중 첫 줄)
    body_lines = body.split("---", 2)[-1].splitlines()
    first_line = next(l for l in body_lines if l.strip()[:2] in ("1.", "- ") or l.strip().startswith("#"))

    r = review_store.patch(target, first_line, first_line + " (checked)", review=fresh_review)
    print("\n=== (2) 읽기 전 patch ===")
    print(r["error"])

    review_store.view(target, review=fresh_review)               # 이번 리뷰에서 읽음 → 기록됨
    r = review_store.patch(target, first_line, first_line + " (checked)", review=fresh_review)
    print("\n=== (2) skill_view 후 재시도 ===")
    print(r["message"])

## Part 5. 전이 실험 — 세션 A에서 배우고 세션 B에서 쓰기

실제 에이전트로 같은 **유형**이지만 **내용이 다른** 작업 두 개를 시킵니다.

| | 세션 A | 세션 B (새 대화) |
|:---|:---|:---|
| 데이터 | `store_sales/` (매장 매출) | `web_traffic/` (웹 방문) |
| 상황 | 원본 CSV 손상, 백업 있음 | 원본 CSV 손상, 백업 있음 (파일명·컬럼 다름) |
| 사용자 | 첫 결과를 보고 **형식을 교정** | 같은 요청, 교정 없음 |

세션 A가 끝나면 리뷰가 스킬을 만들고, 세션 B는 새 대화이므로 시작할 때 그 스킬이 카탈로그에 실립니다. **세션 B가 스킬을 열고, 교정 없이 처음부터 원하는 형식으로 결과를 내는지**를 봅니다.

### 5-1. 실습 데이터 만들기
**왜 필요한가**: 학습한 절차가 "다른 데이터에도 통하는지" 보려면, 구조는 같고 내용은 다른 데이터 두 세트가 필요합니다. 두 폴더 모두 **원본 CSV는 손상**(바이너리 쓰레기 + 잘린 행)되어 있고, **백업 CSV와 스키마(`schema.json`)**가 함께 있습니다. 파일 이름과 컬럼 이름은 일부러 다르게 했습니다.

**볼 것**: `data/` 아래 두 폴더에 각각 원본, 백업, 스키마 3개 파일이 생깁니다.

In [ ]:
LAB = SANDBOX / "data"   # 에이전트가 다룰 데이터 작업 공간

def write_csv(path, header, rows, corrupt=False):
    """CSV 파일을 씁니다. corrupt=True면 중간에 바이너리 쓰레기와 잘린 행을 넣어 손상시킵니다."""
    path.parent.mkdir(parents=True, exist_ok=True)
    lines = [",".join(header)] + [",".join(str(v) for v in r) for r in rows]
    text = "\n".join(lines) + "\n"
    if corrupt:
        text = "\n".join(lines[:3]) + "\n\x00\x00\x07PK\x03garbage\n" + lines[3][:9] + "\n"
    path.write_bytes(text.encode("utf-8"))

# ── 세트 1: 매장 매출 (세션 A에서 사용) ──
sales_header = ["date", "store", "product", "units", "revenue"]
sales_rows = [
    ("2026-07-03", "Gangnam", "latte", 120, 540000), ("2026-07-03", "Hongdae", "latte", 95, 427500),
    ("2026-07-03", "Jamsil", "americano", 140, 490000), ("2026-08-07", "Gangnam", "americano", 160, 560000),
    ("2026-08-07", "Hongdae", "mocha", 60, 330000), ("2026-08-07", "Jamsil", "latte", 110, 495000),
    ("2026-09-04", "Gangnam", "mocha", 70, 385000), ("2026-09-04", "Hongdae", "americano", 130, 455000),
    ("2026-09-04", "Jamsil", "mocha", 50, 275000),
]
write_csv(LAB / "store_sales/sales_2026Q3.csv", sales_header, sales_rows, corrupt=True)   # 손상된 원본
write_csv(LAB / "store_sales/sales_2026Q3.backup.csv", sales_header, sales_rows)          # 정상 백업
(LAB / "store_sales/schema.json").write_text(json.dumps({"columns": sales_header, "numeric": ["units", "revenue"]}))

# ── 세트 2: 웹 트래픽 (세션 B에서 사용) — 파일명·컬럼이 세트 1과 다름 ──
traffic_header = ["day", "page", "visits", "signups"]
traffic_rows = [
    ("2026-07-01", "/home", 5400, 120), ("2026-07-01", "/pricing", 2100, 85), ("2026-07-01", "/blog", 3300, 20),
    ("2026-08-01", "/home", 6100, 140), ("2026-08-01", "/pricing", 2500, 110), ("2026-08-01", "/blog", 2900, 15),
    ("2026-09-01", "/home", 5800, 130), ("2026-09-01", "/pricing", 2700, 120), ("2026-09-01", "/blog", 3600, 25),
]
write_csv(LAB / "web_traffic/traffic_q3.csv", traffic_header, traffic_rows, corrupt=True)   # 손상된 원본
write_csv(LAB / "web_traffic/traffic_q3_bak.csv", traffic_header, traffic_rows)             # 정상 백업
(LAB / "web_traffic/schema.json").write_text(json.dumps({"columns": traffic_header, "numeric": ["visits", "signups"]}))

for p in sorted(LAB.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(LAB))

### 5-2. 에이전트가 쓸 도구 만들기
**왜 필요한가**: 스킬은 "여러 도구를 어떤 순서로 쓰는가"에 대한 절차입니다. 데이터 점검 작업에 필요한 도구 5개를 만들어 에이전트에 줍니다.

| 도구 | 하는 일 |
|:---|:---|
| `list_files(folder)` | 폴더 안 파일 목록 |
| `read_text(path)` | 텍스트 파일 앞부분 읽기 |
| `validate_csv(path, schema_path)` | 스키마 기준 검증 → `ok`, 행 수, 오류 목록 |
| `summarize_csv(path, group_by, metric)` | 그룹별 합계와 총계 |
| `write_report(path, content)` | 보고서(마크다운) 저장 |

보조 함수 두 개도 함께 둡니다.
- `_lab_path()`: 에이전트가 `../` 같은 경로로 데이터 폴더 밖을 읽거나 쓰지 못하게 막습니다.
- `returns_errors`: 도구 안에서 예외가 나면 프로그램을 멈추는 대신 `{"error": ...}`를 돌려줍니다. 손상된 파일을 읽다 실패해도 **에이전트가 오류를 보고 백업으로 넘어가는 판단**을 할 수 있어야 하기 때문입니다. (실제 앱에서는 `ToolErrorHandlerMiddleware`가 같은 역할을 합니다.)

In [ ]:
import csv
import io
import functools

def _lab_path(rel):
    """데이터 작업 공간 안의 경로만 허용합니다."""
    p = (LAB / rel).resolve()
    if not p.is_relative_to(LAB):
        raise ValueError("path must stay inside the data folder")
    return p

def returns_errors(fn):
    """도구 예외를 오류 결과로 돌려줍니다. 에이전트가 실패를 보고 다음 행동을 고를 수 있게 합니다."""
    @functools.wraps(fn)   # 원래 함수의 이름·docstring·인자 정보를 유지 (도구 스키마에 필요)
    def wrapper(*args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})
    return wrapper

# 도구의 docstring은 모델에게 보이는 도구 설명이 됩니다 (그래서 영어로 씀).
@tool
@returns_errors
def list_files(folder: str = "") -> str:
    """List files in a folder of the data workspace (relative path, '' for the root)."""
    base = _lab_path(folder)
    return "\n".join(sorted(str(p.relative_to(LAB)) for p in base.rglob("*") if p.is_file()))

@tool
@returns_errors
def read_text(path: str, max_chars: int = 1500) -> str:
    """Read a text file from the data workspace (truncated to max_chars)."""
    return _lab_path(path).read_text(encoding="utf-8", errors="replace")[:max_chars]

@tool
@returns_errors
def validate_csv(path: str, schema_path: str) -> str:
    """Validate a CSV file against a JSON schema {"columns": [...], "numeric": [...]}. Returns ok, rows, errors."""
    data = _lab_path(path).read_bytes()
    schema = json.loads(_lab_path(schema_path).read_text(encoding="utf-8"))
    errors = []
    if b"\x00" in data:                                    # 바이너리 바이트 → 손상
        errors.append("binary bytes (NUL) found — file is corrupted")
    rows = list(csv.reader(io.StringIO(data.decode("utf-8", errors="replace"))))
    if not rows or rows[0] != schema["columns"]:           # 헤더가 스키마와 다름
        errors.append(f"header mismatch: {rows[0] if rows else None}")
    body = [r for r in rows[1:] if r]
    for i, r in enumerate(body, start=2):
        if len(r) != len(schema["columns"]):               # 열 개수가 다른 행
            errors.append(f"line {i}: expected {len(schema['columns'])} fields, got {len(r)}")
            continue
        for col in schema["numeric"]:                      # 숫자여야 하는 열에 숫자가 아닌 값
            v = r[schema["columns"].index(col)]
            if not v.lstrip("-").isdigit():
                errors.append(f"line {i}: '{col}' is not numeric ({v!r})")
    return json.dumps({"path": path, "ok": not errors, "rows": len(body), "errors": errors[:5]})

@tool
@returns_errors
def summarize_csv(path: str, group_by: str, metric: str) -> str:
    """Sum a numeric metric column grouped by another column. Returns totals per group and the grand total."""
    reader = csv.DictReader(io.StringIO(_lab_path(path).read_text(encoding="utf-8")))
    totals = {}
    for row in reader:
        totals[row[group_by]] = totals.get(row[group_by], 0) + int(row[metric])
    return json.dumps({"group_by": group_by, "metric": metric, "totals": totals, "grand_total": sum(totals.values())})

@tool
@returns_errors
def write_report(path: str, content: str) -> str:
    """Write a markdown report into the data workspace (e.g. reports/x.md)."""
    target = _lab_path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"saved {path} ({len(content)} chars)"

lab_tools = [list_files, read_text, validate_csv, summarize_csv, write_report]
print("✅ 도구:", [t.name for t in lab_tools])

### 5-3. 에이전트 조립
**왜 필요한가**: 실제 앱의 `main_agent`는 미들웨어가 11개라 실습에서 관찰하기 어렵습니다. 여기서는 스킬 학습에 필요한 부분만 남긴 작은 에이전트를 조립합니다.

| 구성 요소 | 역할 |
|:---|:---|
| `lab_mw` (`make_memory_mw`) | 매 턴 `memory_snapshot`을 만들고(Part 2-4), 도구 호출 수를 세고(Part 3), 기준값에 닿은 턴이 끝나면 스킬 리뷰를 실행 |
| `inject_memory_snapshot` (`@dynamic_prompt`) | `lab_mw`가 만든 `memory_snapshot`을 프롬프트 뒤에 붙임. 실제 앱에서는 `PromptAssembler`가 Message 3(`# memorySnapshot`)로 주입 |
| `skill_view`, `skill_manage` 도구 | `lab_mw`가 에이전트에 자동으로 등록 (실습 도구 5개 + 메모리 도구 + 스킬 도구) |

데모에서는 리뷰가 빨리 걸리도록 `interval=5`로 낮춥니다(Hermes 기본값은 10).

`run_turn()`은 한 턴을 실행하고 결과를 요약해 출력하는 편의 함수입니다. 출력 의미:
- 👤 사용자 입력 / 🔧 이번 턴에 호출된 도구 순서 / 🤖 최종 답변 앞부분 / 📈 스킬 리뷰 카운터

In [ ]:
BASE_PROMPT = (
    "You are a data assistant working inside a data workspace. Use the tools to inspect files, "
    "validate data, compute summaries, and save reports. Answer the user in Korean."
)

@dynamic_prompt
def inject_memory_snapshot(request):
    """모델을 호출할 때마다 시스템 프롬프트 = 기본 지시 + 세션 동결 스냅샷(memory_snapshot)."""
    ctx = request.runtime.context
    return BASE_PROMPT + "\n\n" + (getattr(ctx, "memory_snapshot", "") or "")

lab_mw = make_memory_mw(interval=5, review_llm=llm)   # 리뷰에는 실제 LLM 사용
lab_agent = create_agent(
    model=llm,
    tools=lab_tools,
    middleware=[lab_mw, inject_memory_snapshot],      # 순서: 메모리 준비 → 프롬프트 조립
    checkpointer=MemorySaver(),                       # 세션(thread_id)별 대화 유지
    context_schema=AgentContext,
)
lab_store = lab_mw._get_procedural_store(USER_ID)    # 미들웨어가 temporary_user용으로 고른 스토어
print("✅ 스킬 저장 위치:", lab_store.skills_dir.relative_to(project_root))
print("✅ 에이전트 도구:", sorted(t.name for t in lab_tools + lab_mw.tools))

last_ctx = {}   # 세션별 마지막 AgentContext (before_agent가 채운 memory_snapshot을 나중에 확인하려고 보관)

def run_turn(session_id, text):
    """한 턴을 실행하고, 이번 턴의 도구 호출 목록을 돌려줍니다."""
    ctx = lab_context(session_id)
    last_ctx[session_id] = ctx
    out = lab_agent.invoke({"messages": [HumanMessage(content=text)]},
                           config={"configurable": {"thread_id": session_id}}, context=ctx)
    msgs = out["messages"]
    # 대화 전체에서 이번 턴(마지막 사용자 메시지 이후)만 골라 도구 호출 이름을 모읍니다.
    start = max(i for i, m in enumerate(msgs) if isinstance(m, HumanMessage))
    calls = [tc["name"] for m in msgs[start:] if isinstance(m, AIMessage) for tc in (m.tool_calls or [])]
    print(f"👤 {text}")
    print(f"🔧 도구 호출 {len(calls)}회: {' → '.join(calls)}")
    print(f"🤖 {normalize_content(msgs[-1].content)[:400]}")
    print(f"📈 스킬 리뷰 카운터: {lab_mw._skill_iters.get(session_id, 0)} / 기준 {lab_mw.skill_nudge_interval}")
    return calls

### 5-4. 세션 A — 처음 하는 작업
**왜 필요한가**: 형식 지시 없이 작업을 맡겨, 에이전트가 **처음에 어떤 방식으로** 보고서를 쓰는지 기준선을 만듭니다.

**볼 것**: 시작 전 라이브러리에 있는 스킬(Part 1, 4에서 만든 것)은 이번 작업과 무관하므로 열지 않아야 합니다. 도구 호출이 쌓이면서 카운터가 올라갑니다.

In [ ]:
SESSION_A = "lab-session-A"
skills_before_A = {s["name"] for s in lab_store.list_skills()}   # 세션 A 전에 이미 있던 스킬
print("세션 A 시작 전 스킬:", sorted(skills_before_A), "\n")

calls_a1 = run_turn(SESSION_A, "store_sales 폴더 데이터로 정합성 점검 보고서를 만들어서 reports/store_sales_report.md로 저장해줘.")

**왜 필요한가**: 에이전트가 저장한 보고서 파일을 직접 엽니다. 다음 셀에서 사용자가 이 결과를 보고 형식을 교정합니다.

In [ ]:
report_path = LAB / "reports/store_sales_report.md"
print("=== 세션 A 첫 보고서 ===")
print(report_path.read_text(encoding="utf-8") if report_path.exists() else "(보고서 없음)")

### 5-5. 세션 A — 사용자 교정
**왜 필요한가**: 리뷰 프롬프트에서 **사용자 교정은 스킬을 만드는 가장 강한 신호**입니다. 형식 요구 세 가지를 전달합니다. 이 교정이 다음 세션에 전이되는지가 이 실험의 핵심입니다.

**볼 것**: 교정 후 보고서가 세 가지 요구(첫 줄 결론, 마크다운 표, 데이터 출처 절)를 따르는지. 카운터가 기준값 5를 넘는지.

In [ ]:
CORRECTION = (
    "보고서 형식을 이렇게 바꿔서 다시 저장해줘. "
    "1) 첫 줄은 한 문장 결론. "
    "2) 그룹별 합계는 마크다운 표. "
    "3) 마지막에 '데이터 출처' 절을 두고, 원본이 손상돼서 백업을 썼다면 그 사실과 검증 오류를 적어."
)
calls_a2 = run_turn(SESSION_A, CORRECTION)

print("\n=== 세션 A 교정 후 보고서 ===")
print((LAB / "reports/store_sales_report.md").read_text(encoding="utf-8"))

### 5-6. 백그라운드 리뷰 → 스킬 생성
**왜 필요한가**: 스킬 리뷰는 기준값에 닿은 턴이 끝나자마자 백그라운드에서 이미 시작되었습니다. `wait_for_reviews()`로 끝날 때까지 기다린 뒤 결과를 봅니다.

세션 A의 첫 턴(5-4)에서 이미 기준값에 닿았다면 그때 리뷰가 시작되었다가, 교정 턴(5-5)이 시작되면서 취소되고 교정 턴이 끝난 뒤 다시 시작됩니다(Hermes와 같은 동작). 셀을 천천히 실행해 첫 리뷰가 먼저 끝났다면, 교정 전 내용으로 스킬이 먼저 생기고 교정 후 리뷰가 그 스킬을 patch합니다.

**볼 것**
- 리뷰가 끝나면 카운터가 0이 되고 기준값 도달 표시(`_skill_review_pending`)가 비어야 합니다.
- 리뷰 에이전트의 도구 호출 순서(`trace`)와 새로 생긴 스킬의 본문. 교정 내용이 **"요청하면 바꿔라"가 아니라 기본 형식으로** 들어갔는지 확인합니다.

In [ ]:
lab_mw.wait_for_reviews(timeout=300)   # 턴 종료 시 시작된 스킬 리뷰가 끝날 때까지 기다림
print("리뷰 후 카운터:", lab_mw._skill_iters.get(SESSION_A), "| 기준값 도달 표시:", lab_mw._skill_review_pending)

review_a = lab_mw.last_skill_reviews.get(SESSION_A, {})   # 세션 A의 마지막 스킬 리뷰 결과
print("\n=== 리뷰 에이전트의 도구 호출 순서 ===")
for step in review_a.get("trace", []):
    print(" ", step)

learned = [s for s in lab_store.list_skills() if s["name"] not in skills_before_A]   # 이번에 새로 생긴 스킬
for s in learned:
    print(f"\n=== 학습된 스킬: {s['name']} (created_by={s['created_by']}) ===")
    print((lab_store.skills_dir / s["name"] / "SKILL.md").read_text(encoding="utf-8"))
if not learned:
    print("이번 실행에서는 리뷰가 스킬을 만들지 않았습니다. 셀을 다시 실행하거나 5-5의 교정 문장을 더 구체적으로 바꿔 보세요.")

### 5-7. 세션 B — 새 대화, 다른 데이터
**왜 필요한가**: 새 세션(`thread_id`가 다름)이므로 `before_agent`가 스킬 카탈로그를 다시 읽어 `memory_snapshot`에 넣습니다. **형식에 대한 지시는 하지 않고** 같은 유형의 작업만 맡깁니다.

카탈로그에는 세션 A에서 배운 스킬뿐 아니라 Part 1·4에서 만든 스킬(주간 상태 메일, 회의록 정리)도 함께 실립니다. 에이전트가 **여러 스킬 중 이번 작업에 맞는 것을 골라 여는지**도 함께 봅니다.

**볼 것**: 도구 호출이 `skill_view`로 시작하는지, 그리고 프롬프트에 실린 카탈로그와 결과 보고서.

In [ ]:
SESSION_B = "lab-session-B"
calls_b1 = run_turn(SESSION_B, "web_traffic 폴더 데이터로 정합성 점검 보고서를 만들어서 reports/web_traffic_report.md로 저장해줘.")

# 세션 B의 memory_snapshot 중 PROCEDURAL 블록만 잘라서 봅니다 (전체 구조는 Part 2-4 참고).
snapshot_b = last_ctx[SESSION_B].memory_snapshot
start = snapshot_b.find("═" * 46 + "\nPROCEDURAL")
end = snapshot_b.find("</available_skills>") + len("</available_skills>")
print("\n=== 세션 B 시작 시 프롬프트에 실린 스킬 카탈로그 (memory_snapshot) ===")
print(snapshot_b[start:end])

report_path = LAB / "reports/web_traffic_report.md"
print("\n=== 세션 B 보고서 (형식 지시 없음) ===")
print(report_path.read_text(encoding="utf-8") if report_path.exists() else "(보고서 없음)")

### 5-8. 결과 비교
**왜 필요한가**: 보고서를 눈으로 읽는 것만으로는 판단이 흐려질 수 있어, 세션 A에서 교정한 세 가지 형식이 세션 B 보고서에 있는지 코드로 확인합니다.

**볼 것**: 네 항목이 모두 ✅인지, 그리고 사용자 교정 횟수가 1 → 0으로 줄었는지.

In [ ]:
import re

report_b = report_path.read_text(encoding="utf-8") if report_path.exists() else ""
checks = {
    "skill_view로 스킬을 열었다": "skill_view" in calls_b1,
    "마크다운 표가 있다": bool(re.search(r"\|\s*:?-{3,}", report_b)),   # |---| 또는 | :--- | 형태의 구분선
    "'데이터 출처' 절이 있다": "데이터 출처" in report_b,
    "백업 사용을 밝혔다": "백업" in report_b or "backup" in report_b.lower(),
}

print(f"{'':<4}{'세션 A':>14}{'세션 B':>10}")
print(f"{'도구 호출':<10}{len(calls_a1) + len(calls_a2):>10}{len(calls_b1):>10}")
print(f"{'사용자 교정':<9}{1:>10}{0:>10}")
print()
for label, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {label}")

세션 A는 첫 시도와 교정 후 재작성까지 두 번 일했습니다. 세션 B가 스킬을 열고 교정 없이 같은 형식으로 보고서를 냈다면, **세션 A의 교정이 절차로 전이된 것**입니다. 결과는 LLM 실행마다 조금씩 다를 수 있습니다.

도구 호출 수는 줄지 않을 수도 있습니다. 스킬이 절차를 짧게 만드는 경우도 있지만, 이 실습의 스킬은 "백업까지 검증하고, 표에 필요한 지표를 모두 집계한 뒤 작성하라"처럼 **빠뜨리기 쉬운 단계를 추가**하기 때문입니다. 스킬 학습이 줄이는 것은 도구 호출보다 **사용자의 교정 횟수**입니다.

### 💡 모델에 따라 스킬을 열지 않을 수 있습니다
학습 루프가 효과를 내려면 **메인 모델이 카탈로그를 보고 실제로 `skill_view`를 호출해야** 합니다. 같은 스킬과 같은 카탈로그로 세션 B만 반복해 본 결과는 다음과 같았습니다(2026-10 기준).

| 모델 | 세션 B에서 `skill_view` 호출 | 결과 |
|:---|:---|:---|
| `gemini-3.8-flash` (이 프로젝트 기본값) | 첫 동작으로 호출 | 학습된 형식대로 보고서 작성 |
| `openai:gpt-4o-mini`, `openai:gpt-4.1-mini` | 호출하지 않음 | 카탈로그를 무시하고 처음 하는 방식으로 작성 |

Hermes가 카탈로그 안내문을 "**MUST** load it ... **Err on the side of loading**"처럼 강하게 쓰는 이유가 여기에 있습니다. 작은 모델일수록 프롬프트에 목록이 있어도 "이미 할 줄 아는 일"이면 열어 보지 않습니다. 스킬 학습의 효과는 저장소 구조만이 아니라 **메인 모델의 지시 이행 능력**에도 달려 있습니다. Step 0의 `MODEL_NAME`을 바꿔 직접 비교해 보세요.

## Part 6. 자기 개선 — 교정은 새 스킬이 아니라 patch로

세션 B에서 새 요구를 하나 더 교정합니다. 이 교정이 스킬에 반영되는 경로는 두 가지입니다.

| 경로 | 언제 | 근거 |
|:---|:---|:---|
| **포그라운드 patch** | 메인 에이전트가 이번 턴에서 연 스킬이 틀렸다고 판단하면 바로 `skill_manage(action="patch")` | 카탈로그 안내문: "If a loaded skill is wrong or missing a step, fix it with skill_manage" |
| **백그라운드 patch** | 메인 에이전트가 고치지 않았으면, 백그라운드 리뷰가 대화를 다시 보고 patch | 리뷰 프롬프트의 우선순위: 기존 스킬 patch → 보조 파일 → 새 스킬 |

어느 경로든 **새 스킬을 만들지 않고 기존 스킬을 고쳐야** 합니다. 메인 에이전트가 직접 `skill_manage`를 호출하면 카운터가 0이 되므로, 뒤이은 백그라운드 리뷰는 할 일이 없을 수 있습니다.

**왜 아래 셀이 필요한가**: 교정 전후의 스킬 파일을 비교해, 무엇이 어떻게 바뀌었는지 diff로 확인합니다.

**볼 것**
- "이번 턴의 반영 경로"가 포그라운드인지 백그라운드인지
- 스킬 목록이 그대로인지(새 스킬 없음)
- diff에서 `-`는 지워진 줄, `+`는 추가된 줄

In [ ]:
# (1) 교정 전 스킬 본문 저장
before = {s["name"]: (lab_store.skills_dir / s["name"] / "SKILL.md").read_text(encoding="utf-8")
          for s in lab_store.list_skills()}

# (2) 같은 세션 B에서 새 교정 전달
calls_b2 = run_turn(SESSION_B, "표에 '비중(%)' 열도 추가해서 다시 저장해줘. 그룹별 표에는 비중 열이 있어야 해.")
print("\n이번 턴의 반영 경로:", "포그라운드 (메인 에이전트가 skill_manage 호출)" if "skill_manage" in calls_b2
      else "백그라운드 리뷰에 맡김")

# (3) 턴 종료 시 시작된 백그라운드 리뷰가 있으면 끝날 때까지 기다림
lab_mw.wait_for_reviews(timeout=300)
review_b = lab_mw.last_skill_reviews.get(SESSION_B, {})
print("\n=== 리뷰 에이전트의 도구 호출 순서 ===")
for step in review_b.get("trace", []):
    print(" ", step)
print("actions:", review_b.get("actions"))
print("reply  :", review_b.get("reply"))

# (4) 교정 후 스킬 본문과 비교
after = {s["name"]: (lab_store.skills_dir / s["name"] / "SKILL.md").read_text(encoding="utf-8")
         for s in lab_store.list_skills()}
print("\n스킬 목록:", list(before), "→", list(after))
for name in after:
    if name in before and before[name] != after[name]:
        diff = difflib.unified_diff(
            before[name].splitlines(keepends=True), after[name].splitlines(keepends=True),
            fromfile="before", tofile="after",
        )
        print(f"\n=== '{name}' 변경 내용 (diff) ===")
        print("".join(diff))
    elif name not in before:
        print(f"\n=== 새로 생긴 스킬: {name} ===")
        print(after[name])

# (5) 사용 기록: patch_count, last_patched_at 확인
print("\n=== 사용 기록 (.usage.json) ===")
print(json.dumps(lab_store.load_usage(), ensure_ascii=False, indent=2))

확인할 점:
- 스킬 개수가 그대로이고 기존 스킬에 diff가 생겼다면, 교정이 **patch**로 반영된 것입니다.
- 포그라운드에서 이미 고쳤다면 백그라운드 리뷰는 스킬을 열어 보고 "Nothing to save."로 끝나는 것이 정상입니다. 같은 교훈을 두 번 쓰지 않는 것도 리뷰 원칙입니다.
- 에이전트 소유 스킬을 포그라운드에서 고쳐도 소유권(`created_by: agent`)은 유지됩니다.
- 새 스킬이 생겼다면 리뷰가 우선순위를 어긴 것입니다. 이런 중복은 Hermes에서는 Curator의 LLM 통합(consolidation) 단계가 정리합니다. 이 실습에서는 생략했습니다(Part 8).

## Part 7. Curator — 쓰이지 않는 스킬의 생애주기

스킬이 계속 쌓이기만 하면 카탈로그가 길어지고 낡은 절차가 남습니다. Hermes Curator는 **에이전트가 만든 스킬만** 마지막 사용 시각을 기준으로 전이합니다.

```
active ──(14일 미사용)──▶ stale ──(30일 미사용)──▶ archived (.archive/로 이동, 복구 가능)
   ▲                        │
   └──── skill_view로 다시 사용 ┘
```

- 삭제하지 않고 **보관**만 합니다. `restore`로 되돌릴 수 있습니다.
- 사용자 소유 스킬은 건드리지 않습니다.
- Hermes는 유휴 시간이 충분할 때 자동으로 돌지만, 여기서는 `now`를 바꿔 시간을 앞당겨 시연합니다.

**왜 아래 셀이 필요한가**: 지금까지 `temporary_user` 라이브러리에 쌓인 스킬 전체에 Curator를 적용해, 시간이 지나면 상태가 어떻게 바뀌는지 봅니다. 마지막에 보관된 스킬을 모두 복구해 라이브러리를 원래대로 돌려 놓습니다.

**볼 것**
- 20일 후: 에이전트 소유 스킬은 `stale`, 사용자 소유 스킬은 `active` 그대로
- 다시 사용하면 `active`로 복귀
- 40일 후: 에이전트 소유 스킬은 `archived`로 `.archive/` 폴더에 이동, 사용자 소유 스킬만 활성

In [ ]:
from datetime import timedelta
from app.middleware.memory.procedural_store import _now   # 현재 시각(UTC)

curator_store = ProceduralMemoryStore(skills_dir=str(SKILLS_DIR))

def print_states():
    """.usage.json에서 스킬별 상태(state)와 소유자(created_by)를 읽어 출력합니다."""
    for name, record in curator_store.load_usage().items():
        owner = "agent" if record.get("created_by") == "agent" else "user"
        print(f"  {name:<24} state={record.get('state'):<9} owner={owner}")

print("=== 시작 ===")
print_states()

# 시계를 20일 앞으로 → 에이전트 소유 스킬 중 14일 넘게 안 쓴 것은 stale
report = curator_store.apply_lifecycle(now=_now() + timedelta(days=20))
print("\n=== 20일 후 ===")
print("report:", report)
print_states()

# 에이전트 소유 스킬 하나를 다시 사용 → active로 복귀
agent_skills = [n for n, r in curator_store.load_usage().items() if r.get("created_by") == "agent"]
if agent_skills:
    curator_store.view(agent_skills[0])
    print(f"\n=== '{agent_skills[0]}'를 다시 사용 ===")
    print_states()

# 시계를 40일 앞으로 → 30일 넘게 안 쓴 에이전트 스킬은 archived (.archive/로 이동)
report = curator_store.apply_lifecycle(now=_now() + timedelta(days=40))
print("\n=== 40일 후 ===")
print("report:", report)
print_states()
print("활성 스킬:", [s["name"] for s in curator_store.list_skills()])
archive_dir = SKILLS_DIR / ".archive"
print("보관 폴더:", sorted(p.name for p in archive_dir.glob("*")) if archive_dir.exists() else [])

# 보관된 스킬을 모두 복구해 라이브러리를 원래대로
for name in report["archived"]:
    curator_store.restore(name)
print("\n복구 후 활성 스킬:", [s["name"] for s in curator_store.list_skills()])

## Part 8. 교육용으로 생략한 Hermes 운영 기능

이 실습은 **학습 루프**(트리거 → 리뷰 → 스킬 생성/patch → 다음 세션 재사용)와 그 루프를 안전하게 만드는 최소 가드(소유권, read-before-write, 보관)만 구현했습니다. Hermes가 실제 운영에서 추가로 갖추고 있는 기능은 아래와 같습니다. 직접 구현해 볼 확장 과제로도 쓸 수 있습니다.

| 기능 | Hermes에서 하는 일 | 운영에서 필요한 이유 | Hermes 소스 |
|:---|:---|:---|:---|
| **프롬프트 캐시 동일성 포크** | 리뷰 에이전트가 부모의 시스템 프롬프트·도구 목록을 바이트 단위로 그대로 물려받고, 대화를 메시지 그대로 다시 재생 | 리뷰 요청이 부모가 데워 둔 프롬프트 캐시를 재사용해 비용을 줄임(원본 주석상 약 26%) | `agent/background_review.py` `build_cache_parity_fork` |
| **리뷰 토큰 예산** | 리뷰 1회의 입력 토큰 합계 상한(기본 60만), 다른 모델로 보낼 때는 오래된 턴을 요약해 전달 | 긴 세션의 리뷰가 비용 폭주를 일으키지 않도록 | `background_review.py` `_REVIEW_MAX_INPUT_TOKENS_DEFAULT`, `_digest_history` |
| **승인 대기 (write gate)** | 스킬 쓰기를 바로 반영하지 않고 사용자 승인 대기열에 올림 | 스킬은 다음 세션의 행동을 바꾸므로, 조직에 따라 사람 검토가 필요 | `tools/skill_manager_tool.py` `_apply_skill_write_gate`, `tools/write_approval.py` |
| **감사 원장 (ledger)** | 모든 스킬 변경의 전후 내용을 append-only JSONL로 기록 | 언제 누가(어느 리뷰가) 무엇을 바꿨는지 추적·복원 | `tools/skill_ledger.py` |
| **원자적 배치** | `skill_manage`가 작업 배열을 받아 하나라도 실패하면 전부 되돌림 | 여러 스킬에 걸친 정리 작업이 절반만 반영되는 일 방지 | `tools/skill_manager_batch.py` |
| **퍼지 매칭 patch** | 공백·들여쓰기·이스케이프 차이를 흡수해 `old_string`을 찾음 | 모델이 문자열을 정확히 복사하지 못해 patch가 반복 실패하는 일 감소 | `tools/fuzzy_match.py` |
| **보안 스캔·린터** | 새 스킬 폴더의 위험한 코드·비밀값을 검사하고, 형식 문제를 경고 | 스킬의 `scripts/`는 나중에 실행되므로 공급망 위험이 있음 | `skill_manager_tool.py` `_security_scan_skill`, `tools/skill_linter.py` |
| **pin / hub / external / bundled 구분** | 고정된 스킬, 허브 설치 스킬, 외부 폴더 스킬, 기본 탑재 스킬을 각각 보호 | 출처별로 수정 권한이 다름 | `tools/skill_manager_guards.py`, `tools/skill_usage.py` |
| **Curator LLM 통합** | 겹치는 스킬을 LLM이 하나의 상위 스킬로 합치고, 흡수된 스킬은 `absorbed_into`를 남기고 보관 | 리뷰가 비슷한 스킬을 여러 개 만드는 문제를 주기적으로 정리 | `agent/curator.py`, `skill_manager_guards.py` `_curator_consolidation_delete_guard` |
| **Curator 백업** | Curator 실행 전 스킬 폴더 전체를 스냅샷 | 통합 판단이 틀렸을 때 일괄 복구 | `agent/curator_backup.py` |
| **원격 동기화** | 스킬 변경을 디바운스해 원격 저장소로 푸시 | 여러 기기·프로필에서 같은 스킬 라이브러리 사용 | `tools/skills_sync.py` |
| **열람 중복 제거** | 같은 세션에서 바뀌지 않은 스킬을 다시 열면 짧은 안내만 반환 | 같은 본문을 여러 번 컨텍스트에 싣지 않도록 | `tools/skills_tool_dedup.py` |
| **조건부 노출·카테고리 축약** | 플랫폼/환경 조건이 맞는 스킬만 노출, 관련 없는 카테고리는 이름만 표시 | 스킬이 수백 개일 때 카탈로그 크기 관리 | `agent/prompt_builder.py` `_skill_should_show`, `compact_categories` |
| **메모리·스킬 합본 리뷰** | 두 신호가 동시에 오면 한 번의 리뷰로 둘 다 처리 | 리뷰 호출 횟수 절감 | `background_review.py` `_COMBINED_REVIEW_PROMPT` |

이 저장소의 대응:
- 리뷰 에이전트는 `review_llm`으로 새로 만들고, 원문은 텍스트로 렌더링해 전달합니다(캐시 재사용 없음, 원문 16,000자 예산, 모델 호출 최대 16회).
- `skill_manage`는 한 번에 작업 하나를 받는 평면 형태이고, patch는 정확한 문자열 일치만 허용합니다.
- 보호 규칙은 `created_by` 하나로 단순화했습니다. 사람이 작성한 번들 스킬(`app/agents/main_agent/skills/`)은 처음부터 이 저장소 밖에 있으므로 수정 대상이 아닙니다.
- 백그라운드 삭제와 Curator는 보관만 하므로, 백업과 원장이 없어도 복구가 가능합니다.

## Part 9. FAQ

**Q1. `run_skill_review`와 `store.patch`는 무엇이 다른가요? patch 안에서 리뷰를 하나요?**
아닙니다. `run_skill_review`는 **LLM 리뷰 에이전트를 실행해 판단**하는 함수이고, `store.patch`는 **LLM 없이 파일을 고치는** 저장소 메서드입니다. 리뷰 에이전트가 `skill_manage(action="patch")` 도구를 부르면 그 도구가 `store.patch`를 호출합니다. `patch`의 `review=` 인자는 리뷰 실행이 아니라 백그라운드 쓰기라는 권한 표시입니다(Part 4-3).

**Q2. 메인 에이전트도 `skill_manage`를 가지고 있는데, 왜 백그라운드 리뷰가 따로 필요한가요?**
메인 에이전트는 사용자의 요청을 처리하는 데 집중합니다. 일을 끝낸 직후에 "이걸 스킬로 정리해야겠다"고 스스로 판단하는 경우는 드뭅니다. Hermes는 사용자가 명시적으로 시킬 때는 포그라운드에서, 그 외에는 백그라운드 리뷰가 대화를 다시 보고 기록하는 두 경로를 모두 둡니다. 포그라운드에서 새로 만든 스킬은 사용자 소유, 백그라운드에서 만든 스킬은 에이전트 소유가 됩니다.

**Q3. 왜 리뷰를 LLM 한 번 호출 + 정규식 파싱(Semantic 리뷰 방식)이 아니라 별도 에이전트로 하나요?**
스킬은 수십 줄짜리 마크다운이고, 기존 스킬을 읽은 뒤 고쳐야 합니다(read-before-write). 읽기 → 판단 → 쓰기 → 실패 시 재시도가 필요한 작업이라 도구 호출 루프가 맞습니다.

**Q4. 사용자 소유 스킬이 틀렸으면 어떻게 되나요?**
리뷰는 고치지 못하고 마지막 응답에 문제를 적기만 합니다. Hermes에서는 사용자가 `hermes curator adopt <name>`으로 소유권을 넘겨야 자율 수정 대상이 됩니다.

**Q5. 스킬과 Semantic 메모리에 같은 교훈이 중복으로 들어가면요?**
Hermes의 안내문은 "작업 유형에 묶인 교훈은 스킬, 모든 세션에 적용되는 사실만 메모리"로 나눕니다. 리뷰 프롬프트도 사용자 선호가 특정 작업에 관한 것이면 메모리가 아니라 그 작업의 스킬에 넣으라고 지시합니다. 실습 중 메인 에이전트가 형식 선호를 `USER.md`에도 저장했다면, 이 저장소의 `memory` 도구 설명에 그 기준이 없기 때문입니다. 실행 후 `artifacts/users/temporary_user/memory/USER.md`를 열어 확인해 보세요.

**Q6. 리뷰 도중 사용자가 새 질문을 하면요?**
새 턴의 `before_agent`가 실행 중인 리뷰를 취소합니다(Hermes `cancel_background_review_for_live_turn`). 스킬 리뷰 에이전트는 다음 모델 호출 전에 멈추고(`_CancelGuard`), `skill_manage`도 쓰기를 거부합니다. 카운터와 기준값 도달 표시는 유지되어 그 턴이 끝날 때 다시 시도합니다.

## 🧹 Clean-up

**왜 필요한가**: 미들웨어의 백그라운드 스레드를 정리하고, 실습 데이터(CSV, 보고서)를 지웁니다. `temporary_user`의 학습 결과는 남겨 두므로, 파일 탐색기나 에디터에서 `artifacts/users/temporary_user/`를 열어 스킬 파일과 `.usage.json`을 직접 확인해 보세요. 결과까지 지우려면 마지막 줄의 주석을 푸세요.

In [ ]:
lab_mw._review_queue.shutdown()              # 세션 종료 감지 큐(에피소드 추출) 감시 스레드 종료
lab_mw._review_executor.shutdown(wait=False)  # 리뷰 스레드 풀 종료
shutil.rmtree(SANDBOX, ignore_errors=True)   # 실습 데이터 삭제
print(f"✅ 실습 데이터 삭제: {SANDBOX.relative_to(project_root)}")

print(f"\n📂 남겨 둔 학습 결과: {USER_DIR.relative_to(project_root)}/")
for p in sorted(USER_DIR.rglob("*")):
    if p.is_file():
        print("   ", p.relative_to(USER_DIR))

# shutil.rmtree(USER_DIR)   # 학습 결과까지 지우려면 주석 해제